In [ ]:
# Parameters
bronze_schema = "bronze"
silver_schema = "silver"
dbx_sql_source = "lh_databricks.dbo"    # SQL endpoint for the catalog-managed Databricks mirror tables
as_of = ""                  # blank = ref.synthetic_run.generated_for_as_of
stale_feed_hours = 36.0     # DQ-10 threshold
fail_on_check_error = True

# 03 - Silver: validate, standardise, de-duplicate and join

Reads every Bronze table (mirrored ones in place, without a copy), builds one Silver table per cell, validates the
whole layer and only then writes it. Every rule is logged in `silver.dq_rule_summary`; rejected rows go to
`silver.dq_quarantine` with the original record as JSON.

| Rule | Handling |
|---|---|
| DQ-01 duplicates / re-sends | latest `_extracted_at` / export wins |
| DQ-02 unknown SKU, colour or location | quarantine |
| DQ-03 invalid size | quarantine |
| DQ-04 negative on-hand | clamp to 0, keep the raw value, flag |
| DQ-05 returns > cumulative sales | flag, excluded from rates |
| DQ-06 late restatement | merge; restated batch replaces the original |
| DQ-07 missing weather | `weather_status = UNKNOWN/MISSING`, never imputed to 0 |
| DQ-08 mixed time zones | normalise to UTC and market-local date |
| DQ-09 allocated + committed > available | flag over-commit |
| DQ-10 stale feed (> 36 h) | flag in `silver.feed_freshness` |
| DQ-11 cancelled line re-sent as open | status precedence keeps the cancellation |
| DQ-12 owner gaps / duplicates | flag in `silver.decision_owner_coverage` |

In [ ]:
import json, time
import datetime as _dt
from datetime import datetime, timedelta, timezone
from decimal import Decimal
import numpy as np
import pandas as pd
import com.microsoft.spark.fabric   # registers spark.read.synapsesql (catalog-managed Databricks mirror)
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, DecimalType, TimestampType

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

LABEL = "SYNTHETIC"
SILVER_KEYS = {
    "market": ["market_id"], "location": ["location_id"], "product": ["sku_id"], "price": ["style_id", "market_id"],
    "partner": ["partner_id"], "vendor": ["vendor_id"], "transfer_lane": ["from_location_id", "to_location_id"],
    "margin_guardrail": ["market_id", "category", "lifecycle"], "person": ["person_id"],
    "decision_owner": ["category", "market_id", "role", "person_id"], "decision_owner_coverage": ["category", "market_id"],
    "sell_through_daily": ["business_date", "location_id", "sku_id"],
    "digital_intent_daily": ["business_date", "market_id", "style_id", "color"],
    "inventory_position": ["location_id", "sku_id"], "wholesale_order_line": ["order_id", "line_no"],
    "purchase_order_line": ["po_id", "line_no"], "inventory_movement": ["movement_id"],
    "member_engagement": ["business_date", "market_id", "style_id", "color"], "returns": ["return_id", "return_line_no"],
    "weather_daily": ["business_date", "market_id"], "regional_event": ["event_id"], "feed_freshness": ["feed", "market_id"],
    "allocation_decision": ["decision_id"], "stock_transfer_order": ["sto_id", "line_no"],
}


def _date(s):
    return pd.to_datetime(s, errors="coerce").dt.normalize()


def _num(s, kind="float"):
    v = pd.to_numeric(s, errors="coerce")
    return v.astype("Int64") if kind == "int" else v.astype(float)


def _utc(s):
    return pd.to_datetime(s, utc=True, errors="coerce")


def _latest(df, key, order):
    """Keep the latest row per key (by `order`); return the frame and the number of rows removed."""
    out = df.sort_values(order, kind="stable").drop_duplicates(key, keep="last")
    return out, len(df) - len(out)


class DQ:
    """Collects quarantined rows (never silently dropped) and one summary row per rule application."""
    def __init__(self):
        self.quarantine, self.summary = [], []

    def hold(self, df, mask, table, rule, reason, key):
        bad = df[mask]
        if len(bad):
            self.quarantine.append(pd.DataFrame({
                "source_table": table, "rule_id": rule, "reason": reason,
                "business_key": bad[key].astype(str).agg("|".join, axis=1).to_numpy(),
                "record_json": [json.dumps(r, default=str) for r in bad.drop(columns=[c for c in bad.columns if c.startswith("_")]).to_dict("records")],
            }))
        self.note(rule, table, int(mask.sum()), "QUARANTINE", reason)
        return df[~mask]

    def note(self, rule, table, n, handling, detail):
        self.summary.append(dict(rule_id=rule, table_name=table, rows_affected=int(n), handling=handling, detail=detail))


def _label(df):
    df = df.drop(columns=[c for c in df.columns if c.startswith("_") and c != "_ingest_ts"], errors="ignore")
    df["record_label"] = LABEL
    return df.reset_index(drop=True)


def _local_to_utc(ts, tz):
    """DQ-08: values ending in Z or an offset are UTC-aware; naive values are local wall-clock time in `tz`."""
    s = ts.astype("string").str.strip()
    aware = s.str.contains(r"(?:Z|[+-]\d\d:?\d\d)$", regex=True, na=False)
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns, UTC]")
    out[aware] = pd.to_datetime(s[aware], utc=True, errors="coerce")
    naive = ~aware & s.notna()
    for zone, idx in s[naive].groupby(tz[naive]).groups.items():
        local = pd.to_datetime(s.loc[idx], errors="coerce")
        out.loc[idx] = local.dt.tz_localize(zone, ambiguous="NaT", nonexistent="shift_forward").dt.tz_convert("UTC")
    return out, pd.Series(np.where(aware, "UTC", np.where(naive, "LOCAL", None)), index=s.index)

# Bronze contract: mirrored tables are zero-copy shortcuts (notebook 00), file feeds are physical tables (notebook 01)

DBX_TABLES = ("sell_through_daily", "digital_intent_daily")
SQL_TABLES = ("ref.market", "ref.person", "ref.decision_owner", "ref.synthetic_run", "erp.location",
              "erp.product_style", "erp.product_sku", "erp.price", "erp.vendor", "erp.wholesale_partner",
              "erp.wholesale_order", "erp.wholesale_order_line", "erp.purchase_order", "erp.purchase_order_line",
              "erp.margin_guardrail", "erp.regional_event", "wms.inventory_position", "wms.transfer_lane",
              "wms.inventory_movement")
SQL_OPTIONAL_TABLES = ("erp.allocation_decision", "erp.stock_transfer_order")   # write-back tables, may be empty

FILE_TABLES = {"member_engagement": "DS-11", "returns": "DS-12", "weather_daily": "DS-13", "regional_events": "DS-14"}

BRONZE_TABLES = {
    **{f"dbx_{t}": dict(system="databricks_mirror", source=t, ds="DS-01" if "sell" in t else "DS-02") for t in DBX_TABLES},
    **{t.replace(".", "_"): dict(system="azure_sql_mirror", source=t, ds="DS-03..10,19") for t in SQL_TABLES + SQL_OPTIONAL_TABLES},
    **{f"file_{t}": dict(system="synthetic_generator", source=t, ds=ds) for t, ds in FILE_TABLES.items()},
}
OPTIONAL_BRONZE = {n for n, m in BRONZE_TABLES.items() if m["source"] in SQL_OPTIONAL_TABLES}


def bronze_spark_df(name):
    """One Bronze table with lineage columns added at read time. Mirrored Unity Catalog tables are catalog-managed
    Delta, which Spark will not read by path or name, so they are read through the lh_databricks SQL endpoint."""
    meta = BRONZE_TABLES[name]
    if meta["system"] == "databricks_mirror":
        sdf = spark.read.synapsesql(f"{dbx_sql_source}.{meta['source']}")
        obj = f"appareliq.kinetiq_batch.{meta['source']}"
    else:
        sdf = spark.table(f"{bronze_schema}.{name}")
        obj = meta["source"]
    if "_ingest_ts" not in sdf.columns:
        sdf = (sdf.withColumn("_ingest_ts", F.current_timestamp())
                  .withColumn("_source_system", F.lit(meta["system"]))
                  .withColumn("_source_object", F.lit(obj)))
    return sdf

def to_pandas(sdf):
    """DECIMAL -> float, DATE -> datetime64 (midnight), TIMESTAMP -> tz-aware UTC."""
    cols, ts_cols = [], []
    for f in sdf.schema.fields:
        c = F.col(f"`{f.name}`")
        if isinstance(f.dataType, DecimalType):
            c = c.cast("double")
        elif isinstance(f.dataType, DateType):
            c = c.cast("timestamp")
        elif isinstance(f.dataType, TimestampType):
            ts_cols.append(f.name)
        cols.append(c.alias(f.name))
    pdf = sdf.select(*cols).toPandas()
    for c in ts_cols:
        pdf[c] = pd.to_datetime(pdf[c]).dt.tz_localize("UTC")
    return pdf


def _is_date_col(name, s):
    if not pd.api.types.is_datetime64_any_dtype(s) or isinstance(s.dtype, pd.DatetimeTZDtype):
        return False
    if not (name.endswith("_date") or name in ("date", "business_date", "fired_for_date", "as_of_date")):
        return False
    v = s.dropna()
    return v.empty or bool((v == v.dt.normalize()).all())


def to_spark(pdf):
    """pandas -> Spark with stable types: tz-aware -> UTC timestamp, midnight *_date columns -> DATE."""
    pdf = pdf.copy()
    date_cols = []
    for c in pdf.columns:
        s = pdf[c]
        if isinstance(s.dtype, pd.DatetimeTZDtype):
            pdf[c] = s.dt.tz_convert("UTC").dt.tz_localize(None)
        elif _is_date_col(c, s):
            date_cols.append(c)
        elif s.dtype == object:
            kinds = {type(x) for x in s.dropna()}
            if not kinds:
                pdf[c] = s.astype("string")
            elif kinds <= {_dt.date}:
                pass
            elif kinds <= {Decimal}:
                pdf[c] = s.astype(float)
            elif kinds <= {pd.Timestamp, _dt.datetime}:
                pdf[c] = pd.to_datetime(s, utc=True).dt.tz_localize(None)
            elif kinds <= {bool}:
                pdf[c] = s.astype("boolean")
            elif not kinds <= {str}:
                pdf[c] = s.map(lambda x: None if x is None or (isinstance(x, float) and pd.isna(x)) else str(x)).astype("string")
    sdf = spark.createDataFrame(pdf)
    for c in date_cols:
        sdf = sdf.withColumn(c, F.to_date(F.col(c)))
    # pandas NaN/inf would land as IEEE NaN/Infinity, which the SQL endpoint and Direct Lake reject: store NULL
    for c, t in sdf.dtypes:
        if t in ("double", "float"):
            sdf = sdf.withColumn(c, F.when(F.isnan(c) | F.col(c).isin(float("inf"), float("-inf")), F.lit(None))
                                     .otherwise(F.col(c)))
    return sdf


def write_schema(schema, tables):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
    rows = []
    for name, pdf in tables.items():
        to_spark(pdf).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{schema}.{name}")
        rows.append(dict(table_name=f"{schema}.{name}", rows=len(pdf)))
    return pd.DataFrame(rows)

# Check helpers: each check is a row (layer, table, check, passed, detail); assert_all fails the notebook
MAY_BE_EMPTY = {"mart_rebalancing_pairs", "trigger_event", "dq_quarantine", "regional_event",
                "erp_allocation_decision", "erp_stock_transfer_order"}


def _r(layer, table, check, passed, detail=""):
    return dict(layer=layer, table_name=table, check=check, passed=bool(passed), detail=str(detail))


def _generic(layer, tables, keys):
    out = []
    for name, df in tables.items():
        out.append(_r(layer, name, "row_count", len(df) > 0 or name in MAY_BE_EMPTY, f"{len(df)} rows"))
        if "record_label" in df.columns:
            bad = int((df.record_label != LABEL).sum())
            out.append(_r(layer, name, "synthetic_label", bad == 0, f"{bad} unlabelled rows"))
        k = keys.get(name)
        if k and len(df):
            nulls = int(df[k].isna().any(axis=1).sum())
            dups = int(df.duplicated(k).sum())
            out.append(_r(layer, name, "key_not_null", nulls == 0, f"{nulls} null keys {k}"))
            out.append(_r(layer, name, "key_unique", dups == 0, f"{dups} duplicate keys {k}"))
    return out


def _fk(layer, child_name, child, col, parent, pcol=None):
    missing = int((~child[col].dropna().isin(parent[pcol or col])).sum())
    return _r(layer, child_name, f"fk_{col}", missing == 0, f"{missing} orphan {col}")


def assert_all(results):
    failed = results[~results.passed]
    if len(failed):
        raise AssertionError("Checks failed:\n" + failed.to_string(index=False))

In [ ]:
# Read every Bronze table: mirrored ones in place (shortcut / SQL endpoint), file feeds from bronze.file_*
t0 = time.time()
b = {}
for name in BRONZE_TABLES:
    try:
        b[name] = to_pandas(bronze_spark_df(name))
    except Exception:
        if name in OPTIONAL_BRONZE:
            print(f"optional Bronze table {name} not available: skipped")
            continue
        raise
run = b["ref_synthetic_run"]
as_of_ts = pd.Timestamp(as_of or pd.to_datetime(run.generated_for_as_of).max()).normalize()
dq, silver = DQ(), {}
print(f"read {len(b)} Bronze tables, {sum(map(len, b.values())):,} rows in {time.time() - t0:.0f}s; as_of {as_of_ts.date()}")

In [ ]:
# silver.market, location, product, price, partner, vendor, transfer_lane, margin_guardrail, person (masters)
market = b["ref_market"].copy()
loc = b["erp_location"].merge(market[["market_id", "market_name", "region", "currency"]], on="market_id", how="left")
loc["opened_date"] = _date(loc.opened_date)
style = b["erp_product_style"].copy()
style["launch_date"] = _date(style.launch_date)
style_cols = [c for c in style.columns if c not in ("record_label", "size_run") and not c.startswith("_")]
sku = b["erp_product_sku"].merge(style[style_cols], on="style_id", how="left")
sku["is_launch"] = sku.lifecycle.eq("LAUNCH")
price = b["erp_price"].copy()
price["effective_from"] = _date(price.effective_from)
loc_cols = loc[["location_id", "market_id", "location_type"]]
lane = (b["wms_transfer_lane"]
        .merge(loc_cols.rename(columns=lambda c: "from_" + c), on="from_location_id", how="left")
        .merge(loc_cols.rename(columns=lambda c: "to_" + c), on="to_location_id", how="left"))
lane["is_cross_market"] = lane.from_market_id != lane.to_market_id
for name, df in (("location", loc), ("product", sku)):
    before = len(df)
    df.drop_duplicates(SILVER_KEYS[name], keep="last", inplace=True)
    dq.note("DQ-01", name, before - len(df), "DEDUPE", "duplicate master rows")
m = dict(market=market, location=loc, product=sku, price=price, partner=b["erp_wholesale_partner"].copy(),
         vendor=b["erp_vendor"].copy(), transfer_lane=lane, margin_guardrail=b["erp_margin_guardrail"].copy(),
         person=b["ref_person"].copy())
m = {k: _label(v) for k, v in m.items()}
silver.update(m)
print({k: len(v) for k, v in m.items()})

In [ ]:
# silver.decision_owner, decision_owner_coverage (DQ-12: exactly one ACCOUNTABLE owner per category x market)
own = b["ref_decision_owner"].merge(m["person"][["person_id", "display_name", "title", "email"]], on="person_id", how="left")
own = own.drop(columns=["owner_row_id"], errors="ignore")
grid = pd.MultiIndex.from_product([sorted(m["product"].category.dropna().unique()), sorted(m["market"].market_id)],
                                  names=["category", "market_id"]).to_frame(index=False)
acc = own[own.role == "ACCOUNTABLE"].groupby(["category", "market_id"]).agg(
    n_accountable=("person_id", "nunique"), accountable_person_ids=("person_id", lambda s: ",".join(sorted(s))))
cov = grid.merge(acc.reset_index(), on=["category", "market_id"], how="left")
cov["n_accountable"] = cov.n_accountable.fillna(0).astype(int)
cov["gap_type"] = np.where(cov.n_accountable == 0, "NO_OWNER", np.where(cov.n_accountable > 1, "MULTIPLE_OWNERS", None))
cov["dq_owner_gap"] = cov.gap_type.notna()
dq.note("DQ-12", "decision_owner_coverage", int(cov.dq_owner_gap.sum()), "FLAG", "category x market without exactly one accountable owner")
silver["decision_owner"], silver["decision_owner_coverage"] = _label(own), _label(cov)
print(f"owner gaps: {int(cov.dq_owner_gap.sum())}")

In [ ]:
# silver.sell_through_daily (DS-01, Databricks mirror): DQ-03 size, DQ-02 orphans, DQ-06 restatement, DQ-01 re-sends
t = "sell_through_daily"
df = b["dbx_sell_through_daily"].copy()
df["business_date"] = _date(df.business_date)
df["_extracted_at"] = _utc(df._extracted_at)
df["size"] = df["size"].fillna("").astype(str).str.strip()
df["color"] = df["color"].astype(str).str.strip()
df["sku_id"] = df.style_id + "-" + df.color + "-" + df["size"]
sizes = m["product"][["style_id", "size"]].drop_duplicates().assign(_ok=True)
df = df.merge(sizes, on=["style_id", "size"], how="left")
key = ["business_date", "location_id", "sku_id", "_batch_id"]
df = dq.hold(df, df._ok.isna(), t, "DQ-03", "size blank or not in the style size run", key).drop(columns="_ok")
df = dq.hold(df, ~df.sku_id.isin(m["product"].sku_id) | ~df.location_id.isin(m["location"].location_id), t, "DQ-02", "unknown SKU or location", key)
restated = df._batch_id.str.endswith("_r1")
df, removed = _latest(df, SILVER_KEYS[t], ["_extracted_at", "_batch_id"])
dq.note("DQ-06", t, int(restated.sum()), "MERGE", "late restatement replaced the earlier extract (latest _extracted_at wins)")
dq.note("DQ-01", t, removed - int(restated.sum()), "DEDUPE", "re-sent rows; latest extract kept")
df["is_restated"] = df._batch_id.str.endswith("_r1")
df = df.merge(m["location"][["location_id", "market_id"]], on="location_id", how="left")
for c in ("units_sold", "units_returned", "full_price_units"):
    df[c] = df[c].astype("int64")
df["source_batch_id"] = df._batch_id
df["extracted_at"] = df._extracted_at
silver[t] = _label(df)
print(f"silver.{t}: {len(silver[t]):,}")

In [ ]:
# silver.digital_intent_daily (DS-02, Databricks mirror): DQ-02 orphans, DQ-01 re-sends
t = "digital_intent_daily"
df = b["dbx_digital_intent_daily"].copy()
df["business_date"] = _date(df.business_date)
df["_extracted_at"] = _utc(df._extracted_at)
sc = m["product"][["style_id", "color_code"]].drop_duplicates().rename(columns={"color_code": "color"}).assign(_ok=True)
df = df.merge(sc, on=["style_id", "color"], how="left")
bad = df._ok.isna() | ~df.market_id.isin(m["market"].market_id)
df = dq.hold(df, bad, t, "DQ-02", "unknown style-color or market", ["business_date", "market_id", "style_id", "color"]).drop(columns="_ok")
df, removed = _latest(df, SILVER_KEYS[t], ["_extracted_at"])
dq.note("DQ-01", t, removed, "DEDUPE", "re-sent rows; latest extract kept")
df["extracted_at"] = df._extracted_at
silver[t] = _label(df)
print(f"silver.{t}: {len(silver[t]):,}")

In [ ]:
# silver.member_engagement (DS-11, generated feed): typed from strings, DQ-02 orphans, DQ-01 re-sent rows
t = "member_engagement"
df = b["file_member_engagement"].copy()
df["business_date"] = _date(df.business_date)
df["exported_at"] = _utc(df.exported_at)
for c in ("active_members", "app_product_opens", "wishlist_adds", "launch_reminders"):
    df[c] = _num(df[c], "int")
sc = m["product"][["style_id", "color_code"]].drop_duplicates().rename(columns={"color_code": "color"}).assign(_ok=True)
df = df.merge(sc, on=["style_id", "color"], how="left")
bad = df._ok.isna() | df.business_date.isna() | df.app_product_opens.isna()
df = dq.hold(df, bad, t, "DQ-02", "unknown style-color or unparseable row", SILVER_KEYS[t]).drop(columns="_ok")
df, removed = _latest(df, SILVER_KEYS[t], ["exported_at"])
dq.note("DQ-01", t, removed, "DEDUPE", "re-sent file rows; latest export kept")
silver[t] = _label(df)
print(f"silver.{t}: {len(silver[t]):,}")

In [ ]:
# silver.weather_daily (DS-13, generated feed): DQ-08 local -> UTC, DQ-01, DQ-07 complete grid with UNKNOWN (never 0 C)
t = "weather_daily"
df = b["file_weather_daily"].copy()
df["business_date"] = _date(df.business_date)
for c in ("temp_max_c", "temp_min_c", "temp_avg_c", "temp_normal_c", "precip_mm", "precip_normal_mm"):
    df[c] = _num(df[c])
tz = df.market_id.map(m["market"].set_index("market_id").timezone)
df["observed_at_utc"], df["observed_at_source_format"] = _local_to_utc(df.observed_at, tz)
dq.note("DQ-08", t, int(df.observed_at_source_format.eq("LOCAL").sum()), "NORMALISE", "local observation time converted to UTC")
df, removed = _latest(df, SILVER_KEYS[t], ["observed_at_utc"])
dq.note("DQ-01", t, removed, "DEDUPE", "duplicate observations")
start = min(silver["sell_through_daily"].business_date.min(), silver["digital_intent_daily"].business_date.min())
grid = pd.MultiIndex.from_product([pd.date_range(start, as_of_ts - timedelta(days=1), freq="D"), sorted(df.market_id.unique())],
                                  names=["business_date", "market_id"]).to_frame(index=False)
df = grid.merge(df, on=["business_date", "market_id"], how="left")
df["weather_status"] = np.where(df.city.isna(), "MISSING", np.where(df.temp_avg_c.isna(), "UNKNOWN", "OBSERVED"))
df["dq_weather_unknown"] = df.weather_status.ne("OBSERVED")
df["city"] = df.city.fillna(df.groupby("market_id").city.transform(lambda s: s.dropna().iloc[0] if s.notna().any() else None))
df["temp_anomaly_c"] = df.temp_avg_c - df.temp_normal_c
df["precip_ratio"] = np.where(df.precip_normal_mm > 0, df.precip_mm / df.precip_normal_mm, np.nan)
dq.note("DQ-07", t, int(df.dq_weather_unknown.sum()), "FLAG_UNKNOWN", "missing weather kept as UNKNOWN, never imputed to 0")
silver[t] = _label(df)
print(f"silver.{t}: {len(silver[t]):,}")

In [ ]:
# silver.inventory_position (DS-05, ERP/WMS mirror): DQ-02 orphans, DQ-04 negative stock, DQ-09 over-commit
t = "inventory_position"
df = b["wms_inventory_position"].copy()
df = dq.hold(df, ~df.sku_id.isin(m["product"].sku_id) | ~df.location_id.isin(m["location"].location_id), t, "DQ-02",
             "unknown SKU or location", ["location_id", "sku_id"])
df["snapshot_ts"] = pd.to_datetime(df.snapshot_ts)
df["snapshot_date"] = df.snapshot_ts.dt.normalize()
df["on_hand_units_raw"] = df.on_hand_units.astype("int64")
df["dq_negative_stock"] = df.on_hand_units_raw < 0
df["on_hand_units"] = df.on_hand_units_raw.clip(lower=0)
dq.note("DQ-04", t, int(df.dq_negative_stock.sum()), "CLAMP_AND_FLAG", "negative on-hand clamped to 0")
df["committed_wholesale_units"] = df.reserved_units.astype("int64")
df["dq_overcommit"] = (df.allocated_units + df.reserved_units) > (df.on_hand_units + df.in_transit_units)
dq.note("DQ-09", t, int(df.dq_overcommit.sum()), "FLAG", "allocated + committed exceeds on hand + in transit")
df["available_units"] = (df.on_hand_units - df.allocated_units - df.reserved_units).clip(lower=0)
df = df.merge(m["location"][["location_id", "market_id", "location_type"]], on="location_id", how="left")
silver[t] = _label(df)
print(f"silver.{t}: {len(silver[t]):,}")

In [ ]:
# silver.wholesale_order_line (DS-07): DQ-11 status precedence CANCELLED > RESERVED > OPEN; cancelled header cancels all lines
hdr = b["erp_wholesale_order"].copy()
for c in ("order_date", "requested_ship_date"):
    hdr[c] = _date(hdr[c])
ln = b["erp_wholesale_order_line"].merge(hdr.drop(columns=["record_label"] + [c for c in hdr.columns if c.startswith("_")]),
                                         on="order_id", how="left")
rank = {"CANCELLED": 3, "RESERVED": 2, "OPEN": 1}
top = ln.assign(_r=ln.line_status.map(rank).fillna(0)).groupby(["order_id", "sku_id"])._r.transform("max")
eff = pd.Series(top.map({v: k for k, v in rank.items()}), index=ln.index)
ln["effective_status"] = np.where(ln.order_status.eq("CANCELLED"), "CANCELLED", eff)
ln["dq_cancel_resend"] = ln.line_status.ne(ln.effective_status) & ln.effective_status.eq("CANCELLED")
dq.note("DQ-11", "wholesale_order_line", int(ln.dq_cancel_resend.sum()), "STATUS_PRECEDENCE", "cancelled line re-sent as open; cancellation kept")
ln["committed_units"] = np.where(ln.effective_status.isin(["OPEN", "RESERVED"]), ln.ordered_units, 0)
ln["cancelled_units"] = np.where(ln.effective_status.eq("CANCELLED") & ~ln.dq_cancel_resend, ln.ordered_units, 0)
silver["wholesale_order_line"] = _label(ln)
print(f"silver.wholesale_order_line: {len(ln):,}")

In [ ]:
# silver.purchase_order_line (DS-08): slip days = current - original promise
hdr = b["erp_purchase_order"].copy()
for c in ("po_date", "original_promise_date", "current_promise_date"):
    hdr[c] = _date(hdr[c])
ln = b["erp_purchase_order_line"].merge(hdr.drop(columns=["record_label"] + [c for c in hdr.columns if c.startswith("_")]),
                                        on="po_id", how="left")
ln["slip_days"] = (ln.current_promise_date - ln.original_promise_date).dt.days.astype("int64")
ln = ln.merge(m["location"][["location_id", "market_id"]].rename(columns={"location_id": "ship_to_location_id"}),
              on="ship_to_location_id", how="left")
silver["purchase_order_line"] = _label(ln)
print(f"silver.purchase_order_line: {len(ln):,}")

In [ ]:
# silver.inventory_movement (DS-09)
df = b["wms_inventory_movement"].copy()
df["movement_ts"] = pd.to_datetime(df.movement_ts)
df["movement_date"] = df.movement_ts.dt.normalize()
silver["inventory_movement"] = _label(df)
print(f"silver.inventory_movement: {len(df):,}")

In [ ]:
# silver.returns (DS-12, generated feed): DQ-02, DQ-01, DQ-08 local -> UTC + local date, DQ-05 returns > cumulative sales
t = "returns"
df = b["file_returns"].copy()
df["exported_at"] = _utc(df.exported_at)
df["units"] = _num(df.units, "int")
df["refund_local"] = _num(df.refund_local)
df["original_purchase_date"] = _date(df.original_purchase_date)
bad = ~df.sku_id.isin(m["product"].sku_id) | ~df.location_id.isin(m["location"].location_id) | df.units.isna()
df = dq.hold(df, bad, t, "DQ-02", "unknown SKU or location", SILVER_KEYS[t])
df, removed = _latest(df, SILVER_KEYS[t], ["exported_at"])
dq.note("DQ-01", t, removed, "DEDUPE", "re-exported return lines; latest export kept")
df = df.merge(m["location"][["location_id", "market_id", "timezone"]], on="location_id", how="left")
df["return_ts_utc"], df["return_ts_source_format"] = _local_to_utc(df.return_ts, df.timezone)
local = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
for zone, idx in df.groupby("timezone").groups.items():
    local.loc[idx] = df.loc[idx, "return_ts_utc"].dt.tz_convert(zone).dt.tz_localize(None).dt.normalize()
df["return_local_date"] = local
dq.note("DQ-08", t, int(df.return_ts_source_format.eq("LOCAL").sum()), "NORMALISE", "local wall-clock timestamps converted to UTC + market-local date")
sold = silver["sell_through_daily"].groupby(["location_id", "sku_id"]).units_sold.sum().rename("cum_units_sold")
ret = df.groupby(["location_id", "sku_id"]).units.sum().rename("cum_units_returned")
chk = pd.concat([sold, ret], axis=1).fillna(0)
over = chk[chk.cum_units_returned > chk.cum_units_sold].index
df["dq_returns_exceed_sales"] = pd.MultiIndex.from_frame(df[["location_id", "sku_id"]]).isin(over)
dq.note("DQ-05", t, int(df.dq_returns_exceed_sales.sum()), "FLAG", "returns exceed cumulative sales; excluded from rates")
silver[t] = _label(df.drop(columns=["timezone"]))
print(f"silver.{t}: {len(silver[t]):,}")

In [ ]:
# silver.regional_event (DS-14 feed reconciled with the ERP event calendar): DQ-01 superseded versions
f = b["file_regional_events"].copy()
f["version"] = _num(f.version, "int")
f["published_at"] = _utc(f.published_at)
f, removed = _latest(f, ["event_id"], ["version", "published_at"])
dq.note("DQ-01", "regional_event", removed, "DEDUPE", "superseded event versions")
for c in ("start_date", "end_date"):
    f[c] = _date(f[c])
f["expected_attendance"] = _num(f.expected_attendance, "int")
f["demand_uplift_hint"] = _num(f.demand_uplift_hint)
e = b["erp_regional_event"][["event_id", "demand_uplift"]].rename(columns={"demand_uplift": "erp_demand_uplift"})
ev = f.merge(e, on="event_id", how="outer", indicator=True)
ev["in_erp_calendar"] = ev._merge.isin(["both", "right_only"])
ev["in_external_feed"] = ev._merge.isin(["both", "left_only"])
silver["regional_event"] = _label(ev.drop(columns="_merge"))
print(f"silver.regional_event: {len(ev):,}")

In [ ]:
# silver.allocation_decision, stock_transfer_order (DS-10: ERP decision history + agent write-back; empty until seeded)
# History rows (proposal_id HIST-*) are seeded by Databricks notebook 02_load_erp_wms_sql; later rows are agent write-backs
if len(b.get("erp_allocation_decision", ())):
    d = b["erp_allocation_decision"].copy()
    d["decision_id"] = d.decision_id.astype(str).str.lower()
    d["decided_at_utc"] = _utc(d.decided_at_utc)
    d["decision_source"] = np.where(d.proposal_id.str.startswith("HIST-"), "ERP_HISTORY", "AGENT_WRITEBACK")
    silver["allocation_decision"] = _label(d)
    sto = b.get("erp_stock_transfer_order", pd.DataFrame())
    if len(sto):
        sto = sto.copy()
        sto["decision_id"] = sto.decision_id.astype(str).str.lower()
        sto["units"] = _num(sto.units, "int")
        sto["requested_ship_date"] = _date(sto.requested_ship_date)
        sto["created_at_utc"] = _utc(sto.created_at_utc)
        silver["stock_transfer_order"] = _label(sto)
    print(f"silver.allocation_decision: {len(d):,}; silver.stock_transfer_order: {len(sto):,}")
else:
    print("no allocation decisions in the ERP yet: decision history skipped")


In [ ]:
# silver.feed_freshness (DQ-10): hours since the latest delivered business date per feed x market, measured at as-of 06:00 UTC
now = (as_of_ts + pd.Timedelta(hours=6)).tz_localize("UTC")
wx = silver["weather_daily"]
feeds = (("sell_through_daily", silver["sell_through_daily"]), ("digital_intent_daily", silver["digital_intent_daily"]),
         ("member_engagement", silver["member_engagement"]), ("weather_daily", wx[wx.weather_status.eq("OBSERVED")]))
rows = []
for feed, df in feeds:
    last = df.groupby("market_id").business_date.max()
    for mk in sorted(m["market"].market_id):
        ld = last.get(mk, pd.NaT)
        expected_at = (ld + pd.Timedelta(days=1, hours=3)).tz_localize("UTC") if pd.notna(ld) else pd.NaT
        lag = (now - expected_at).total_seconds() / 3600 if pd.notna(expected_at) else None
        rows.append(dict(feed=feed, market_id=mk, last_business_date=ld, freshness_lag_hours=lag,
                         dq_stale_feed=bool(lag is None or lag > stale_feed_hours)))
fresh = pd.DataFrame(rows)
fresh["freshness_lag_hours"] = fresh.freshness_lag_hours.astype(float)
dq.note("DQ-10", "feed_freshness", int(fresh.dq_stale_feed.sum()), "FLAG", f"feed older than {stale_feed_hours:.0f} h")
silver["feed_freshness"] = _label(fresh)
display(fresh[fresh.dq_stale_feed])

In [ ]:
# silver.dq_quarantine, dq_rule_summary, run_context
q = pd.concat(dq.quarantine, ignore_index=True) if dq.quarantine else pd.DataFrame(
    columns=["source_table", "rule_id", "reason", "business_key", "record_json"])
q["quarantined_at"] = pd.Timestamp(datetime.combine(as_of_ts.date(), _dt.time(6), tzinfo=timezone.utc))
silver["dq_quarantine"] = _label(q)
silver["dq_rule_summary"] = _label(pd.DataFrame(dq.summary))
silver["run_context"] = _label(pd.DataFrame([dict(as_of_date=as_of_ts, silver_built_for=str(as_of_ts.date()),
                                                  source_run_id=str(run.run_id.iloc[0]))]))
display(silver["dq_rule_summary"])

In [ ]:
# Validation: row counts, lineage, primary keys (not null, unique), foreign keys, null handling and DQ invariants
out = []
for name, df in b.items():
    out.append(_r("bronze", name, "row_count", len(df) > 0 or name in MAY_BE_EMPTY, f"{len(df)} rows"))
    for c in ("_ingest_ts", "_source_system"):
        out.append(_r("bronze", name, f"lineage_{c}", c in df.columns and df[c].notna().all(), c))
missing = sorted(set(BRONZE_TABLES) - set(b) - MAY_BE_EMPTY)
out.append(_r("bronze", "*", "all_tables_present", not missing, f"missing: {missing}"))

s = silver
out += _generic("silver", s, SILVER_KEYS)
for n in ("sell_through_daily", "inventory_position", "returns"):
    out.append(_fk("silver", n, s[n], "sku_id", s["product"]))
    out.append(_fk("silver", n, s[n], "location_id", s["location"]))
inv, w, wol = s["inventory_position"], s["weather_daily"], s["wholesale_order_line"]
unk = w[w.weather_status != "OBSERVED"]
out += [
    _r("silver", "inventory_position", "DQ-04_no_negative_on_hand", (inv.on_hand_units >= 0).all(), "clamped"),
    _r("silver", "weather_daily", "DQ-07_unknown_not_imputed", unk.temp_avg_c.isna().all(), f"{len(unk)} unknown rows keep null temperature"),
    _r("silver", "returns", "DQ-08_utc_parsed", s["returns"].return_ts_utc.notna().all(), "all timestamps in UTC"),
    _r("silver", "sell_through_daily", "units_non_negative", (s["sell_through_daily"].units_sold >= 0).all(), ""),
    _r("silver", "wholesale_order_line", "DQ-11_cancelled_not_committed", (wol[wol.effective_status == "CANCELLED"].committed_units == 0).all(), ""),
    _r("silver", "dq_rule_summary", "dq_rules_recorded", s["dq_rule_summary"].rule_id.nunique() >= 8,
       ",".join(sorted(s["dq_rule_summary"].rule_id.unique()))),
]
if "allocation_decision" in s:
    ad = s["allocation_decision"]
    acc = s["decision_owner"][s["decision_owner"].role == "ACCOUNTABLE"]
    cat = s["product"][["style_id", "category"]].drop_duplicates()
    who = ad.merge(cat, on="style_id", how="left").merge(acc[["category", "market_id", "person_id"]],
                   left_on=["category", "market_id", "approver_person_id"], right_on=["category", "market_id", "person_id"], how="left")
    # The demo operator (persona 'operator') may decide any case through the agent gateway, which checks that authority
    op = who.decision_source.eq("AGENT_WRITEBACK") & who.approver_person_id.eq("operator")
    out += [
        _fk("silver", "allocation_decision", ad, "style_id", s["product"]),
        _fk("silver", "allocation_decision", ad, "market_id", s["market"]),
        _r("silver", "allocation_decision", "approver_is_accountable_owner", (who.person_id.notna() | op).all(),
           f"{int((who.person_id.isna() & ~op).sum())} decisions not approved by the accountable owner; "
           f"{int(op.sum())} by the demo operator via the agent gateway"),
    ]
    if "stock_transfer_order" in s:
        sto = s["stock_transfer_order"]
        out += [
            _fk("silver", "stock_transfer_order", sto, "decision_id", ad),
            _fk("silver", "stock_transfer_order", sto, "sku_id", s["product"]),
            _fk("silver", "stock_transfer_order", sto, "from_location_id", s["location"], "location_id"),
            _fk("silver", "stock_transfer_order", sto, "to_location_id", s["location"], "location_id"),
            _r("silver", "stock_transfer_order", "only_for_approved_decisions",
               sto.decision_id.isin(ad[ad.decision.isin(["APPROVED", "MODIFIED"])].decision_id).all(), "no transfer without approval (AG-02)"),
        ]
checks = pd.DataFrame(out)
print(f"{int(checks.passed.sum())}/{len(checks)} checks passed")
display(checks[~checks.passed] if (~checks.passed).any() else checks.tail(10))
if fail_on_check_error:
    assert_all(checks)

In [ ]:
# Write Silver only after every check passed
t0 = time.time()
written = write_schema(silver_schema, {**silver, "dq_check_result": checks.assign(record_label=LABEL)})
print(f"wrote {len(written)} Silver tables in {time.time() - t0:.0f}s")
display(written)